> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 02. 손 landmark와 제스처 상태 — 25분

**예상 시간: 25분**

이 노트북은 영상 처리를 처음 배우는 비전공 학부생을 기준으로 합니다. 설명을 외우기보다 **입력 → 배열 → 변환 → 저장 → 재검증** 순서를 따라가세요.

목표는 21개 좌표에서 네 손가락의 상대 거리를 계산해 OPEN_PALM/FIST/POINT/UNKNOWN/NO_HAND을 명시적으로 만드는 것입니다.

필요한 실행·저장 규칙은 이 노트북에서 다시 제공합니다. [07.Mediapipe README](README.md)와 [용어 찾아보기](GLOSSARY.md)는 추가 참고 자료입니다. TODO는 JupyterLab 파일 편집기에서 저장하고, 미완성 검증이 실패하는 것은 정상입니다. 참고 정답 셀은 강사가 안내하는 시점에 펼칩니다.

1시간 중 손 핵심 25분 뒤 합성·저장 10분과 결과 점검 5분을 포함합니다.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

MediaPipe는 21개 손 관절을 내고, 우리가 만든 함수가 그 좌표에서 손가락 상태를 계산합니다. 모델 출력과 규칙 판단을 분리하면 실제 카메라 오류인지 거리비 조건의 오류인지 좁힐 수 있습니다.

### 전체 과정에서 내 코드가 하는 일

```text
영상
  ↓
21개 특징점
  ↓
거리비 TODO
  ↓
extended·label·status
  ↓
합성 검증
  ↓
녹화·실제 손 비교
```

임계값 직전·정확한 등호·직후와 손 없음·불확실을 구현합니다. 제공 runtime이 학생 파일을 자동으로 읽는다고 가정하지 않고 각 결과의 실행 경로를 확인합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** NO_HAND와 UNKNOWN은 둘 다 움직이지 않으니 같나요?

**설명:** 행동은 같을 수 있지만 원인은 다릅니다. NO_HAND는 손 입력이 없고 UNKNOWN은 입력이 있으나 규칙으로 확정하지 못한 상태입니다. 복구할 때 카메라·검출과 규칙·가림 중 어디를 볼지 달라집니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**MediaPipe 준비:** Ollama·ROS node는 필요 없습니다. 배포 자산은 `Data/models/07.Mediapipe/{face_landmarker.task,hand_landmarker.task}`, `Data/images/07.Mediapipe/person_open_palm.png`, `Data/videos/07.Mediapipe/person_open_palm.avi`입니다. 첫 Python 셀의 환경 검사가 모델·입력 해시와 실제 검출을 확인합니다. 파일이 없거나 검사에 실패하면 해당 배포 경로·수업 이미지를 확인한 뒤 재실행합니다. 입력은 사전 제공 파일이며 다른 노트북의 실행 결과를 요구하지 않습니다.

얼굴·손 카메라 시연 전에도 **컨테이너 JupyterLab Terminal** 에서 `cd /workspace` 후 `/opt/course/bin/python 07.Mediapipe/scripts/check_environment.py`로 준비를 확인할 수 있습니다. USB·GUI는 로컬 한 곳에서 실행하고 두 수강생이 함께 봅니다. 결과 영상은 `Data/outputs/07.Mediapipe`, 상태 JSON은 `Data/logs/07.Mediapipe`에 남습니다. 같은 이름의 재실행 결과가 갱신되므로 비교할 파일은 먼저 별도 이름으로 보관합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[06.OpenCV/02_colors_drawing.ipynb의 1절](../06.OpenCV/02_colors_drawing.ipynb)에 BGR·RGB, [01_face_gaze.ipynb의 1절](01_face_gaze.ipynb)에 모델 좌표와 규칙 판정의 구분이 있습니다. 필요한 내용은 OpenCV BGR을 제공 `TasksDetector`가 RGB로 바꾸고, 모델이 좌표를 찾은 뒤 **우리 함수가 좌표에서 상태를 정한다** 는 것입니다. 손은 21개 점이며 None은 미검출입니다. 얼굴 TODO·얼굴 결과 파일은 필요 없습니다. 손목·손가락 번호와 거리 계산은 아래 TODO에 모두 다시 제시합니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–5절과 단원 완료 점검** → [3일차 슬라이드 32쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s027) · [배포 PDF 32쪽](../슬라이드/3일차.pdf#page=32)에서 40쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 흐름 그림

![파랑은 PIP, 초록은 tip, 주황은 손목입니다. 아래 정답 해설의 거리 예제를 이 점들에 대응해 보세요.](../교재그림/hand-landmarks.svg)

파랑은 PIP, 초록은 tip, 주황은 손목입니다. 아래 정답 해설의 거리 예제를 이 점들에 대응해 보세요.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 손 상태·미검출 횟수와 결과 영상을 저장합니다.
- **단원 마지막 연결 질문:** 한 장면에서 성공한 것과 여러 사람·가림·조명에서 쓸 수 있는 것을 구분합니다. 손 제스처는 이 사례의 선택 입력이며 부품 검사 모델은 아닙니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import json, subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '07.Mediapipe/common').is_dir())
COURSE = ROOT / '07.Mediapipe'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.contracts import check
from common.geometry import estimate_gaze, classify_hand
from common.synthetic import make_eye, make_hand, draw_hand
from common.runtime import OUTPUT_DIR
probe = subprocess.run([sys.executable, str(COURSE/'scripts/check_environment.py')], text=True, capture_output=True, timeout=60)
print(probe.stdout)
if probe.stderr: print('MediaPipe runtime log:\n'+probe.stderr[-1500:])
probe.check_returncode()


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 필요한 landmark만 먼저 읽습니다

손목은 0번입니다. 검지·중지·약지·소지의 PIP/tip은 `(6,8)`, `(10,12)`, `(14,16)`, `(18,20)`입니다. tip의 손목 거리가 PIP 거리의 1.15배보다 크면 편 상태로 봅니다.

절대 y 좌표 대신 거리 비율을 쓰면 손의 화면 크기와 회전에 대한 민감도가 일부 줄어듭니다. 하지만 손가락이 카메라 방향을 향하거나 가려지면 2D 거리만으로 올바르게 분류할 수 없습니다.


In [ ]:
# 구분: 수업 핵심 셀
for expected in ('OPEN_PALM','FIST','POINT','UNKNOWN'):
    points = make_hand(expected)
    result = classify_hand(points)
    print(expected, '→', result)
assert classify_hand(None)['label'] == 'NO_HAND'


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. 상태표를 코드로 옮깁니다

<!-- procedure:lesson-25-818110:start -->
![모델 좌표와 제스처 규칙 연결](../교재그림/절차/lesson-25.svg)

모델은 점을 검출하고 규칙이 라벨을 만듭니다. 손 모양의 분류와 ROS 이동 명령을 혼동하지 않습니다.
<!-- procedure:lesson-25-818110:end -->


| 조건 | label | 후속 제어에서의 예 |
|---|---|---|
| 네 손가락 모두 펴짐 | OPEN_PALM | 정지 요청 후보 |
| 모두 접힘 | FIST | 정지/선택 후보 |
| 검지만 펴짐 | POINT | 진행 요청 후보 |
| 그 밖의 조합 | UNKNOWN | 동작하지 않음 |
| landmark 없음 | NO_HAND | 입력 없음, 동작하지 않음 |

실제 로봇 동작 매핑은 ROS 단원에서 허용 명령·속도·유효 시간을 추가한 뒤 결정합니다. 여기서는 인식 label이 곧 actuator 명령이라고 가정하지 않습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 수업 핵심 TODO — [gesture.py](exercises/gesture.py)

수정 파일: [exercises/gesture.py](exercises/gesture.py)

입력 None, 정확히 21개인지, finite 좌표인지 검사하고 입력 목록을 변경하지 마세요. PIP/tip 거리의 경계에서 `>`와 `>=` 중 계약에 맞는 연산을 사용합니다.

JupyterLab 파일 편집기에서 TODO를 구현하고 **저장** 한 뒤 아래 검증을 실행합니다. 검증은 저장된 모듈을 새 이름으로 다시 읽으므로 일반적으로 kernel 재시작은 필요 없습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `classify_hand(landmarks, extension_ratio=1.15)`는 21개 점의 x,y 또는 None을 받아 `status,label,extended`의 dict를 반환합니다. None → `{'status':'NO_HAND','label':'NO_HAND','extended':[]}`; 검지만 펴짐 → `{'status':'OK','label':'POINT','extended':['INDEX']}`.
**작성 흐름 힌트:** None/개수/좌표·1<ratio<2 검사 → 손목0에서 PIP `(6,10,14,18)`, tip `(8,12,16,20)`까지의 2차원 거리 → strict `>`로 INDEX/MIDDLE/RING/PINKY 목록 구성 → 4개 OPEN_PALM, 0개 FIST, INDEX만 POINT, 나머지 UNKNOWN.
**주의할 경계:** 정확히 tip거리=PIP거리×ratio이면 펴짐에 포함하지 않습니다. 엄지는 이번 네 손가락 규칙에 포함하지 않으며 입력을 뒤집거나 정렬하지 않습니다.
**설명할 지점:** “UNKNOWN은 미검출 NO_HAND와 다릅니다. 좌표는 있지만 이 규칙으로 이름을 정하지 못한 경우입니다.”
**거리 식·오류·통과 기준:** 각 점은 최소 x,y 두 좌표이고 float로 읽은 두 값이 유한해야 합니다. `math.hypot(x1-x0,y1-y0)` 또는 `((x1-x0)**2+(y1-y0)**2)**0.5`로 거리를 구합니다. 필요한 `math`는 직접 import합니다. 21개가 아닌 점 목록, NaN·Inf 좌표, `1<extension_ratio<2` 위반은 `ValueError`입니다. None이면 좌표 계산 전 NO_HAND를 반환합니다. 손가락 이름의 반환 순서는 INDEX→MIDDLE→RING→PINKY입니다. 검증은 `check('gesture','exercise')`이며 네 label의 status·extended 전체, None·20개 점·NaN·원본 보존, 거리의 등호 전후와 잘못된 ratio를 확인합니다. 카메라 runtime은 제공 `common.geometry`를 사용하므로 학습자 TODO의 수정과 실제 검출 모델 변경을 혼동하지 않습니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE/'exercises/gesture.py').read_text())


In [ ]:
# 구분: 수업 핵심 셀
check('gesture', 'exercise')
print('learner_verification=PASS')


> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

### 참고 정답

강사가 안내하는 시점에 정답과 해설을 비교합니다. 정답은 별도 파일이며 exercise를 덮어쓰지 않습니다. 구현이 한 글자씩 같을 필요는 없고 입력·출력·오류 처리 계약이 같으면 됩니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE/'solutions/gesture.py').read_text())


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/gesture.py](solutions/gesture.py)의 `classify_hand`입니다. 손목 0에서 PIP(손가락 중간 관절)와 tip(끝)까지 거리를 비교합니다. 검지 (6,8), 중지 (10,12), 약지 (14,16), 소지 (18,20)를 쓰고 엄지는 이 규칙에서 제외합니다.

1. landmarks=None이면 NO_HAND입니다. 잘못된 길이나 비유한 좌표는 정상 손 상태를 만들지 않고 오류로 알립니다. 모델 미검출과 깨진 입력은 다릅니다.
2. `_point`로 x·y를 읽고 `_distance`로 두 점의 거리를 계산합니다. 두 축 차이를 함께 사용하므로 단순한 y 높이 비교와 다릅니다.
3. `tip_distance > pip_distance * extension_ratio`이면 그 손가락 이름을 extended에 추가합니다. 기본 배수는 1.15입니다.
4. 네 손가락 모두면 OPEN_PALM, 하나도 없으면 FIST, 검지만이면 POINT, 나머지 조합은 UNKNOWN입니다. UNKNOWN은 손이 있지만 지정 조합으로 판정하기 어렵다는 뜻입니다.

| 검지의 PIP 거리 | tip 거리 | 판정 |
|---:|---:|---|
| 0.20 | 0.30 | 0.30 > 0.23이므로 펴짐 |
| 0.20 | 0.23 | 수학적으로 같은 경계이므로 엄격한 `>`는 거짓 |
| 0.20 | 0.18 | 접힘 |

실수 연산의 정확한 경계는 검증 셀의 고정 입력으로 확인합니다. 네 거리 중 검지만 펴졌다면 POINT입니다. 두 손가락만 펴졌다고 임의 방향 명령을 만들지 않습니다. 화면 크기·회전 영향은 일부 줄지만 가림·원근에는 실패할 수 있습니다.

**중요한 실행 구분:** 자신의 함수는 학습자 검증에서 읽습니다. run_synthetic.py와 실제 카메라 TasksDetector는 제공된 common.geometry 함수를 사용합니다. 학생 TODO의 임계값 수정이 실제 카메라에 자동으로 반영되는 구조는 아닙니다.

**확인 질문:** NO_HAND와 UNKNOWN을 왜 구분하고, 둘을 downstream에서 임의의 이동으로 바꾸면 안 될까요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/gesture.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 21개 좌표 → 손목과 관절/끝의 거리 비교 → 손 상태.
PIP는 손가락 중간 관절, tip은 끝입니다. 엄지를 제외한 네 손가락을 판정합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
import math
from typing import Sequence

Point = tuple[float, float]


NO_HAND = {"status": "NO_HAND", "label": "NO_HAND", "extended": []}


def _point(value: Sequence[float], name: str) -> Point:
    if len(value) < 2:
        raise ValueError(f"{name}에는 x, y가 필요합니다.")
    x, y = float(value[0]), float(value[1])
    if not (math.isfinite(x) and math.isfinite(y)):
        raise ValueError(f"{name} 좌표는 유한한 숫자여야 합니다.")
    return x, y


def _distance(a: Point, b: Point) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])


def classify_hand(
    landmarks: Sequence[Sequence[float]] | None,
    *,
    extension_ratio: float = 1.15,
) -> dict[str, object]:
    """21개 손 landmark의 거리 비율로 네 손가락 상태를 분류한다.

    손목(0)에서 tip까지의 거리가 PIP까지 거리보다 충분히 길면 편 것으로
    판단한다. 이 교육용 규칙은 회전에 비교적 강하지만 가림·원근·손가락을
    옆으로 편 자세에는 실패할 수 있다.
    """
    if landmarks is None:
        return dict(NO_HAND)
    if len(landmarks) != 21:
        raise ValueError(f"손 landmark는 정확히 21개여야 합니다: {len(landmarks)}")
    if not (1.0 < extension_ratio < 2.0):
        raise ValueError("extension_ratio는 1과 2 사이여야 합니다.")
    points = [_point(value, f"landmark[{index}]") for index, value in enumerate(landmarks)]
    wrist = points[0]
    names = ("INDEX", "MIDDLE", "RING", "PINKY")
    pip_indices = (6, 10, 14, 18)
    tip_indices = (8, 12, 16, 20)
    # 엄지를 제외한 네 손가락만 비교하며 정확한 경계는 펴짐에서 제외합니다.
    extended = [
        name
        for name, pip, tip in zip(names, pip_indices, tip_indices)
        if _distance(points[tip], wrist) > _distance(points[pip], wrist) * extension_ratio
    ]
    if len(extended) == 4:
        label = "OPEN_PALM"
    elif len(extended) == 0:
        label = "FIST"
    elif extended == ["INDEX"]:
        label = "POINT"
    else:
        label = "UNKNOWN"
    return {"status": "OK", "label": label, "extended": extended}
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
check('gesture', 'solution')
print('solution_verification=PASS')


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 카메라 없는 저장 검증

합성 좌표는 MediaPipe 검출 정확도를 평가하지 않습니다. 좌표 뒤의 분류 코드, label 집합, Data 저장을 언제나 같은 입력으로 확인합니다. 환경 셀은 실제 두 `.task` 모델에서 빈 이미지의 미검출과 AI 생성 양성 이미지의 `LOOK_CENTER`·`OPEN_PALM`을 모두 확인했습니다.


In [ ]:
# 구분: 수업 핵심 셀
run = subprocess.run([sys.executable, str(COURSE/'scripts/run_synthetic.py')],
                     text=True, capture_output=True, timeout=60)
print(run.stdout)
if run.stderr: print(run.stderr)
run.check_returncode()
records = json.loads((OUTPUT_DIR/'synthetic_results.json').read_text())
assert len(records) == 7
assert {r['label'] for r in records if r['kind']=='gesture'} == {'OPEN_PALM','FIST','POINT','UNKNOWN'}


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 4. 실제 양성 녹화 입력

고정 입력 검증은 AI 생성 양성 PNG를 640×480, 6 FPS, 24 frame으로 만든 고정 AVI를 사용합니다. 실제 손 움직임을 평가하는 영상은 아니지만 MediaPipe 검출, 반복 처리, annotation writer와 JSON 집계를 함께 검증합니다.


In [ ]:
# 구분: 수업 핵심 셀
positive_video = ROOT/'Data/videos/07.Mediapipe/person_open_palm.avi'
command = [sys.executable, str(COURSE/'scripts/run_tasks.py'), '--mode', 'hand',
           '--source', str(positive_video), '--max-frames', '24', '--headless']
run = subprocess.run(command, text=True, capture_output=True, timeout=90)
print(run.stdout)
if run.stderr: print(run.stderr[-1500:])
run.check_returncode()
report = json.loads((ROOT/'Data/logs/07.Mediapipe/hand_results.json').read_text())
assert report['frames'] == 24 and report['counts'] == {'OPEN_PALM': 24}
assert report['output_frame_count'] in (0, 24)


### 저장된 처리 결과를 노트북에서 보기

바로 위 양성 녹화 입력 셀을 실행한 뒤 아래 셀을 실행합니다. AVI는 브라우저에서 직접 재생되지 않는 경우가 있어, 저장된 24프레임을 GIF로 변환해 표시합니다. 카메라를 열거나 MediaPipe를 다시 실행하지 않습니다.

손 특징점과 `OPEN_PALM` 표시를 확인하세요. 같은 AI 생성 이미지를 반복한 고정 영상이므로 움직임은 거의 없습니다. 이 결과로 실제 손 움직임의 추적 성능을 평가하지 않습니다. 미리보기는 `Data/outputs/07.Mediapipe/hand_positive_preview.gif`에 저장됩니다.


In [ ]:
# 저장된 양성 입력 처리 결과만 표시합니다. 카메라나 모델을 다시 실행하지 않습니다.
from pathlib import Path
import json
import cv2
from PIL import Image as PILImage
from IPython.display import Image, display

preview_root = ROOT
preview_report = json.loads((preview_root / 'Data/logs/07.Mediapipe/hand_results.json').read_text())
expected_source = preview_root / 'Data/videos/07.Mediapipe/person_open_palm.avi'
if Path(preview_report['source']).resolve() != expected_source.resolve():
    raise ValueError('현재 결과는 양성 녹화 입력의 결과가 아닙니다. 바로 위 양성 녹화 입력 셀을 먼저 실행하세요.')
preview_video = Path(preview_report['output'])
preview_capture = cv2.VideoCapture(str(preview_video))
preview_frames = []
try:
    if not preview_capture.isOpened():
        raise FileNotFoundError(f'저장 영상을 열 수 없습니다: {preview_video}')
    preview_fps = preview_capture.get(cv2.CAP_PROP_FPS)
    if not 0 < preview_fps <= 120:
        preview_fps = 6.0
    for _ in range(24):
        ok, frame = preview_capture.read()
        if not ok:
            break
        # OpenCV는 BGR, Pillow와 브라우저는 RGB 색 순서를 사용합니다.
        preview_frames.append(PILImage.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
finally:
    preview_capture.release()
if len(preview_frames) != 24:
    raise ValueError(f'양성 녹화 결과는 24프레임이어야 합니다: {len(preview_frames)}')
preview_gif = preview_video.with_name('hand_positive_preview.gif')
preview_frames[0].save(preview_gif, save_all=True, append_images=preview_frames[1:],
                       duration=round(1000 / preview_fps), loop=0)
print('프레임 수:', len(preview_frames), '| 상태별 횟수:', preview_report['counts'])
display(Image(filename=str(preview_gif), format='gif'))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 5. 실제 입력으로 바꾸기

로컬 USB 카메라는 로컬 컨테이너의 Terminal에서 작업 위치를 먼저 고정한 뒤 실행합니다.

```bash
cd /workspace
/opt/course/bin/python 07.Mediapipe/scripts/run_tasks.py \
  --mode hand --source camera --mirror --max-frames 300
```

- 선택 재생 영상: `--source /workspace/Data/videos/07.Mediapipe/person_open_palm.avi --headless`를 사용합니다.
- 직접 촬영한 파일은 다른 이름으로 Data에 저장하고 제공 원본을 보존합니다.

한 frame의 label만 보지 말고 JSON의 label 횟수, 미검출 구간, 입력 내용과 annotation을 함께 봅니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 손 TODO·합성 저장·실제 입력 결과를 구분; 얼굴 시연은 31쪽에서 먼저 진행

[3일차 슬라이드 32쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s027) · [배포 PDF 32쪽](../슬라이드/3일차.pdf#page=32)로 돌아가 **32–40쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

## 자습 — 엄지, handedness와 시간 필터

엄지는 다른 네 손가락과 움직임 축이 달라 x 좌표 비교가 흔하지만, 왼손/오른손과 mirror에 따라 부호가 바뀝니다. handedness 결과를 명시적으로 함께 전달한 뒤 규칙을 추가하세요. 최근 N frame 다수결을 넣을 때 NO_HAND가 이어지면 이전 gesture를 언제 만료할지도 반드시 정합니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 07 단원 완료 점검

- 모델 검출과 좌표 규칙이 별도 단계임을 설명할 수 있습니다.
- UNKNOWN과 NO_HAND을 다른 상태로 유지합니다.
- 손 TODO의 경계·오류 검증과 합성 저장 7건을 통과했습니다. 얼굴 TODO는 자습입니다.
- USB camera/GUI 실습은 로컬에서 함께 진행합니다. 녹화/headless는 독립 재생·자습에서 선택합니다.
- 실제 USB 카메라는 한 프로세스만 열고 종료 시 capture와 detector를 닫습니다.
